[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnCrabs/GeoEducationGR/blob/main/ΒΑΛΕ_ΕΔΩ_ΤΗ_ΔΙΑΔΡΟΜΗ/ΟΝΟΜΑ.ipynb)

# Εφαρμογή 1: Γεωμετρικές οντότητες στο επίπεδο

Σε αυτή την εφαρμογή φτιάχνουμε από την αρχή τρεις βασικές κλάσεις για γεωμετρία στο επίπεδο:

| Κλάση | Τι αναπαριστά |
|---|---|
| `Point2D` | Ένα σημείο με συντεταγμένες $(X, Y)$ |
| `Line2D` | Ένα ευθύγραμμο τμήμα που ορίζεται από δύο σημεία |
| `Polygon` | Ένα πολύγωνο που ορίζεται από μια σειρά κορυφών |

Κάθε κλάση έχει:
- **υπολογισμούς** (απόσταση, αζιμούθιο, μήκος, εμβαδόν, κέντρο βάρους κ.ά.),
- μια συνάρτηση **`describe()`** που επιστρέφει μια πλήρη περιγραφή της οντότητας σε κείμενο,
- μια συνάρτηση **`plot()`** για να τη σχεδιάσουμε.

Χρησιμοποιούμε μόνο την τυπική βιβλιοθήκη της Python (`math`) και το `matplotlib`, που υπάρχει ήδη στο Google Colab, οπότε δεν χρειάζεται καμία εγκατάσταση.

## 0. Σύμβαση συντεταγμένων και αζιμουθίου

Ακολουθούμε τη σύμβαση της Τοπογραφίας:

- Ο άξονας **Y** δείχνει τον **Βορρά** και ο άξονας **X** την **Ανατολή**.
- Το **αζιμούθιο** $\alpha_{AB}$ της διεύθυνσης $A \rightarrow B$ μετριέται από τον Βορρά (άξονας Y), **δεξιόστροφα**:

$$
\alpha_{AB} = \operatorname{atan2}(\Delta X_{AB},\ \Delta Y_{AB}), \qquad \Delta X_{AB} = X_B - X_A,\quad \Delta Y_{AB} = Y_B - Y_A
$$

- Το αποτέλεσμα φέρνεται στο διάστημα $[0, 400)$ **grad** (ή $[0^\circ, 360^\circ)$).
- Η **απόσταση** είναι $S_{AB} = \sqrt{\Delta X_{AB}^2 + \Delta Y_{AB}^2}$.

Προσοχή: η `math.atan2(y, x)` της Python δέχεται πρώτα το «y». Εδώ της δίνουμε πρώτα το $\Delta X$, επειδή η γωνία μετριέται από τον άξονα Y.

In [ ]:
import math
import matplotlib.pyplot as plt

# Ανοχή για συγκρίσεις αριθμών κινητής υποδιαστολής
# (π.χ. 0.1 + 0.2 δεν είναι ακριβώς 0.3 στον υπολογιστή)
TOL = 1e-9

# Μετατροπές γωνιών
RAD_TO_GRAD = 200.0 / math.pi
RAD_TO_DEG = 180.0 / math.pi


def convert_angle(angle_rad, unit="grad"):
    # Μετατρέπει γωνία από ακτίνια στη μονάδα που ζητάμε: 'grad', 'deg' ή 'rad'
    if unit == "grad":
        return angle_rad * RAD_TO_GRAD
    if unit == "deg":
        return angle_rad * RAD_TO_DEG
    if unit == "rad":
        return angle_rad
    raise ValueError(f"Άγνωστη μονάδα γωνίας: {unit!r}. Επιτρέπονται 'grad', 'deg', 'rad'.")


print("Έτοιμο!")

## 1. Η κλάση `Point2D`

Ένα σημείο έχει συντεταγμένες $X, Y$ και προαιρετικά ένα **όνομα** (π.χ. `"A"`, `"Σ1"`).

Βασικές μέθοδοι:

| Μέθοδος | Τι κάνει |
|---|---|
| `distance_to(p)` | Απόσταση από άλλο σημείο |
| `delta_to(p)` | Διαφορές $\Delta X, \Delta Y$ προς άλλο σημείο |
| `azimuth_to(p, unit)` | Αζιμούθιο προς άλλο σημείο |
| `midpoint(p)` | Μέσο του τμήματος με άλλο σημείο |
| `translate(dx, dy)` | Νέο σημείο μετατοπισμένο κατά $dx, dy$ |
| `rotate(angle, center)` | Νέο σημείο στραμμένο γύρω από ένα κέντρο |
| `quadrant()` | Σε ποιο τεταρτημόριο βρίσκεται |
| `describe()` | Πλήρης περιγραφή σε κείμενο |
| `plot(ax)` | Σχεδίαση |

Οι **ειδικές μέθοδοι** (dunder methods) `__repr__`, `__str__`, `__eq__` και `__iter__` κάνουν την κλάση να συμπεριφέρεται «φυσικά» στην Python: σωστή εκτύπωση, σύγκριση με `==` και αποσυσκευασία `x, y = p`.

In [ ]:
class Point2D:
    """Σημείο στο επίπεδο με συντεταγμένες (X, Y) και προαιρετικό όνομα."""

    def __init__(self, x, y, name=None):
        self.x = float(x)
        self.y = float(y)
        self.name = name

    # ------------------------------------------------ Ειδικές μέθοδοι
    def __repr__(self):
        # Τεχνική αναπαράσταση: από αυτή μπορεί να ξαναφτιαχτεί το αντικείμενο
        return f"Point2D(x={self.x}, y={self.y}, name={self.name!r})"

    def __str__(self):
        # Φιλική αναπαράσταση για εκτύπωση με print()
        label = self.name if self.name else "P"
        return f"{label}({self.x:.3f}, {self.y:.3f})"

    def __eq__(self, other):
        # Δύο σημεία είναι ίσα αν οι συντεταγμένες τους συμπίπτουν (με ανοχή TOL)
        if not isinstance(other, Point2D):
            return NotImplemented
        return (math.isclose(self.x, other.x, abs_tol=TOL)
                and math.isclose(self.y, other.y, abs_tol=TOL))

    def __iter__(self):
        # Επιτρέπει: x, y = p
        yield self.x
        yield self.y

    # ------------------------------------------------ Βασικές πληροφορίες
    def to_tuple(self):
        return (self.x, self.y)

    def quadrant(self):
        """Επιστρέφει το τεταρτημόριο (1-4) ή περιγραφή αν το σημείο είναι πάνω σε άξονα."""
        if math.isclose(self.x, 0, abs_tol=TOL) and math.isclose(self.y, 0, abs_tol=TOL):
            return "αρχή των αξόνων"
        if math.isclose(self.x, 0, abs_tol=TOL):
            return "πάνω στον άξονα Y"
        if math.isclose(self.y, 0, abs_tol=TOL):
            return "πάνω στον άξονα X"
        if self.x > 0 and self.y > 0:
            return 1
        if self.x < 0 and self.y > 0:
            return 2
        if self.x < 0 and self.y < 0:
            return 3
        return 4

    # ------------------------------------------------ Σχέσεις με άλλο σημείο
    def delta_to(self, other):
        """Επιστρέφει (ΔX, ΔY) από αυτό το σημείο προς το other."""
        return (other.x - self.x, other.y - self.y)

    def distance_to(self, other):
        """Ευκλείδεια απόσταση S = sqrt(ΔX² + ΔY²)."""
        dx, dy = self.delta_to(other)
        return math.hypot(dx, dy)

    def azimuth_to(self, other, unit="grad"):
        """Αζιμούθιο από τον Βορρά (άξονας Y), δεξιόστροφα.

        unit: 'grad' (0-400), 'deg' (0-360) ή 'rad' (0-2π)
        """
        dx, dy = self.delta_to(other)
        if math.isclose(dx, 0, abs_tol=TOL) and math.isclose(dy, 0, abs_tol=TOL):
            raise ValueError("Τα δύο σημεία συμπίπτουν: το αζιμούθιο δεν ορίζεται.")
        a = math.atan2(dx, dy)          # Προσοχή: πρώτα ΔX, μετά ΔY
        if a < 0:
            a += 2 * math.pi            # Φέρνουμε τη γωνία στο [0, 2π)
        return convert_angle(a, unit)

    def midpoint(self, other, name=None):
        """Μέσο του τμήματος που ενώνει τα δύο σημεία."""
        return Point2D((self.x + other.x) / 2, (self.y + other.y) / 2, name)

    # ------------------------------------------------ Μετασχηματισμοί (επιστρέφουν ΝΕΟ σημείο)
    def translate(self, dx, dy, name=None):
        """Μετατόπιση κατά (dx, dy)."""
        return Point2D(self.x + dx, self.y + dy, name or self.name)

    def rotate(self, angle_deg, center=None, name=None):
        """Στροφή κατά angle_deg μοίρες αριστερόστροφα (μαθηματική φορά) γύρω από το center.

        Αν δεν δοθεί center, η στροφή γίνεται γύρω από την αρχή (0, 0).
        """
        cx, cy = (center.x, center.y) if center else (0.0, 0.0)
        t = math.radians(angle_deg)
        x0, y0 = self.x - cx, self.y - cy
        xr = x0 * math.cos(t) - y0 * math.sin(t)
        yr = x0 * math.sin(t) + y0 * math.cos(t)
        return Point2D(xr + cx, yr + cy, name or self.name)

    # ------------------------------------------------ Περιγραφή και σχεδίαση
    def describe(self):
        """Επιστρέφει πλήρη περιγραφή του σημείου σε κείμενο."""
        origin = Point2D(0, 0)
        lines = [
            f"Σημείο: {self.name if self.name else '(χωρίς όνομα)'}",
            f"  X = {self.x:.3f}",
            f"  Y = {self.y:.3f}",
            f"  Τεταρτημόριο: {self.quadrant()}",
            f"  Απόσταση από την αρχή (0, 0): {self.distance_to(origin):.3f}",
        ]
        if self != origin:
            lines.append(f"  Αζιμούθιο από την αρχή: {origin.azimuth_to(self):.4f} grad"
                         f" ({origin.azimuth_to(self, 'deg'):.4f}°)")
        return "\n".join(lines)

    def plot(self, ax=None, color="tab:red", show_label=True, **kwargs):
        """Σχεδιάζει το σημείο. Επιστρέφει τον άξονα (ax) για να προσθέσουμε κι άλλα στοιχεία."""
        if ax is None:
            _, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(self.x, self.y, color=color, zorder=3, **kwargs)
        if show_label:
            ax.annotate(str(self), (self.x, self.y), textcoords="offset points",
                        xytext=(5, 5), fontsize=9)
        ax.set_aspect("equal")
        ax.grid(True, linestyle=":")
        ax.set_xlabel("X (Ανατολή)")
        ax.set_ylabel("Y (Βορράς)")
        return ax

### Παραδείγματα με `Point2D`

In [ ]:
A = Point2D(100.0, 200.0, "A")
B = Point2D(150.0, 260.0, "B")

print(A)              # χρησιμοποιεί την __str__
print(repr(A))        # χρησιμοποιεί την __repr__
print()
print(A.describe())

In [ ]:
print(f"ΔX, ΔY από A προς B : {A.delta_to(B)}")
print(f"Απόσταση S_AB       : {A.distance_to(B):.3f}")
print(f"Αζιμούθιο α_AB      : {A.azimuth_to(B):.4f} grad = {A.azimuth_to(B, 'deg'):.4f}°")
print(f"Αζιμούθιο α_BA      : {B.azimuth_to(A):.4f} grad  (διαφέρει κατά 200 grad)")
print(f"Μέσο του AB         : {A.midpoint(B, 'M')}")
print(f"A == Point2D(100, 200)? {A == Point2D(100, 200)}")

x, y = A              # αποσυσκευασία χάρη στην __iter__
print(f"x = {x}, y = {y}")

In [ ]:
# Μετασχηματισμοί: επιστρέφουν νέο σημείο, το αρχικό δεν αλλάζει
A_moved = A.translate(10, -20, "A'")
A_rot = A.rotate(90, center=B, name="A''")

ax = A.plot()
B.plot(ax, color="tab:blue")
A_moved.plot(ax, color="tab:green")
A_rot.plot(ax, color="tab:purple")
ax.set_title("Σημεία και μετασχηματισμοί")
plt.show()

## 2. Η κλάση `Line2D`

Η `Line2D` αναπαριστά ένα **ευθύγραμμο τμήμα** από το σημείο `start` στο σημείο `end`. Έχει φορά, άρα το $AB$ έχει διαφορετικό αζιμούθιο από το $BA$.

Βασικές μέθοδοι:

| Μέθοδος | Τι κάνει |
|---|---|
| `dx`, `dy`, `length` | $\Delta X$, $\Delta Y$, μήκος |
| `azimuth(unit)` | Αζιμούθιο της διεύθυνσης start → end |
| `slope()` | Κλίση $\Delta Y / \Delta X$ (άπειρο για κατακόρυφη) |
| `equation()` | Συντελεστές της ευθείας $aX + bY + c = 0$ |
| `midpoint()`, `point_at(t)` | Μέσο και σημείο σε ποσοστό $t \in [0, 1]$ του μήκους |
| `contains_point(p)` | Αν ένα σημείο βρίσκεται πάνω στο τμήμα |
| `distance_to_point(p)` | Ελάχιστη απόσταση σημείου από το τμήμα |
| `is_parallel(l)`, `is_perpendicular(l)` | Παραλληλία και καθετότητα |
| `intersection(l)` | Σημείο τομής με άλλο τμήμα (ή `None`) |
| `reversed()` | Το ίδιο τμήμα με αντίθετη φορά |
| `describe()`, `plot(ax)` | Περιγραφή και σχεδίαση |

Για την τομή δύο τμημάτων $P + t\,\mathbf{r}$ και $Q + u\,\mathbf{s}$ χρησιμοποιούμε το εξωτερικό γινόμενο $\mathbf{a} \times \mathbf{b} = a_x b_y - a_y b_x$:

$$
t = \frac{(Q - P) \times \mathbf{s}}{\mathbf{r} \times \mathbf{s}}, \qquad
u = \frac{(Q - P) \times \mathbf{r}}{\mathbf{r} \times \mathbf{s}}
$$

Τα τμήματα τέμνονται αν $0 \le t \le 1$ και $0 \le u \le 1$.

In [ ]:
def cross(ax_, ay_, bx_, by_):
    """Εξωτερικό γινόμενο 2D διανυσμάτων: a × b = ax·by − ay·bx."""
    return ax_ * by_ - ay_ * bx_


class Line2D:
    """Ευθύγραμμο τμήμα από το σημείο start στο σημείο end."""

    def __init__(self, start, end, name=None):
        if not isinstance(start, Point2D) or not isinstance(end, Point2D):
            raise TypeError("Τα άκρα του τμήματος πρέπει να είναι αντικείμενα Point2D.")
        if start == end:
            raise ValueError("Τα άκρα συμπίπτουν: το τμήμα θα είχε μηδενικό μήκος.")
        self.start = start
        self.end = end
        # Αν δεν δοθεί όνομα αλλά τα σημεία έχουν, το όνομα γίνεται π.χ. "AB"
        if name is None and start.name and end.name:
            name = f"{start.name}{end.name}"
        self.name = name

    # ------------------------------------------------ Ειδικές μέθοδοι
    def __repr__(self):
        return f"Line2D(start={self.start!r}, end={self.end!r}, name={self.name!r})"

    def __str__(self):
        label = self.name if self.name else "L"
        return f"{label}: {self.start} → {self.end}"

    def __eq__(self, other):
        # Ίσα αν έχουν τα ίδια άκρα με την ίδια φορά
        if not isinstance(other, Line2D):
            return NotImplemented
        return self.start == other.start and self.end == other.end

    # ------------------------------------------------ Βασικά μεγέθη
    @property
    def dx(self):
        return self.end.x - self.start.x

    @property
    def dy(self):
        return self.end.y - self.start.y

    @property
    def length(self):
        return math.hypot(self.dx, self.dy)

    def azimuth(self, unit="grad"):
        """Αζιμούθιο της διεύθυνσης start → end."""
        return self.start.azimuth_to(self.end, unit)

    def slope(self):
        """Κλίση ΔY/ΔX. Για κατακόρυφο τμήμα επιστρέφει math.inf."""
        if math.isclose(self.dx, 0, abs_tol=TOL):
            return math.inf
        return self.dy / self.dx

    def angle_with_x_axis(self, unit="deg"):
        """Γωνία με τον άξονα X, αριστερόστροφα (μαθηματική φορά), στο [0, 2π)."""
        a = math.atan2(self.dy, self.dx)
        if a < 0:
            a += 2 * math.pi
        return convert_angle(a, unit)

    def equation(self):
        """Συντελεστές (a, b, c) της ευθείας aX + bY + c = 0 που περνά από το τμήμα."""
        a = self.dy
        b = -self.dx
        c = -(a * self.start.x + b * self.start.y)
        return a, b, c

    def equation_str(self):
        a, b, c = self.equation()
        return f"{a:.3f}·X {b:+.3f}·Y {c:+.3f} = 0"

    # ------------------------------------------------ Σημεία πάνω στο τμήμα
    def midpoint(self, name=None):
        return self.start.midpoint(self.end, name)

    def point_at(self, t, name=None):
        """Σημείο σε ποσοστό t του μήκους (t=0 → start, t=1 → end)."""
        return Point2D(self.start.x + t * self.dx, self.start.y + t * self.dy, name)

    def reversed(self):
        """Το ίδιο τμήμα με αντίθετη φορά (end → start)."""
        name = self.name[::-1] if self.name else None
        return Line2D(self.end, self.start, name)

    # ------------------------------------------------ Σχέσεις με σημεία
    def _projection_t(self, p):
        # Παράμετρος t της ορθής προβολής του p πάνω στην ευθεία του τμήματος
        return ((p.x - self.start.x) * self.dx + (p.y - self.start.y) * self.dy) / self.length ** 2

    def distance_to_point(self, p):
        """Ελάχιστη απόσταση του σημείου p από το τμήμα (όχι από την άπειρη ευθεία)."""
        t = min(1.0, max(0.0, self._projection_t(p)))
        return p.distance_to(self.point_at(t))

    def contains_point(self, p, tol=1e-6):
        """True αν το σημείο p βρίσκεται πάνω στο τμήμα (με ανοχή tol)."""
        return self.distance_to_point(p) <= tol

    # ------------------------------------------------ Σχέσεις με άλλο τμήμα
    def is_parallel(self, other):
        return math.isclose(cross(self.dx, self.dy, other.dx, other.dy), 0,
                            abs_tol=TOL * self.length * other.length)

    def is_perpendicular(self, other):
        dot = self.dx * other.dx + self.dy * other.dy
        return math.isclose(dot, 0, abs_tol=TOL * self.length * other.length)

    def intersection(self, other, name=None):
        """Σημείο τομής δύο τμημάτων ή None αν δεν τέμνονται.

        Για παράλληλα (ή συνευθειακά) τμήματα επιστρέφει None.
        """
        rx, ry = self.dx, self.dy
        sx, sy = other.dx, other.dy
        denom = cross(rx, ry, sx, sy)
        if math.isclose(denom, 0, abs_tol=TOL * self.length * other.length):
            return None
        qpx = other.start.x - self.start.x
        qpy = other.start.y - self.start.y
        t = cross(qpx, qpy, sx, sy) / denom
        u = cross(qpx, qpy, rx, ry) / denom
        eps = 1e-12
        if -eps <= t <= 1 + eps and -eps <= u <= 1 + eps:
            return self.point_at(t, name)
        return None

    # ------------------------------------------------ Περιγραφή και σχεδίαση
    def describe(self):
        """Επιστρέφει πλήρη περιγραφή του τμήματος σε κείμενο."""
        slope = self.slope()
        slope_txt = "άπειρη (κατακόρυφο τμήμα)" if math.isinf(slope) else f"{slope:.4f}"
        lines = [
            f"Ευθύγραμμο τμήμα: {self.name if self.name else '(χωρίς όνομα)'}",
            f"  Αρχή : {self.start}",
            f"  Τέλος: {self.end}",
            f"  ΔX = {self.dx:.3f},  ΔY = {self.dy:.3f}",
            f"  Μήκος: {self.length:.3f}",
            f"  Αζιμούθιο: {self.azimuth():.4f} grad ({self.azimuth('deg'):.4f}°)",
            f"  Γωνία με τον άξονα X: {self.angle_with_x_axis():.4f}°",
            f"  Κλίση: {slope_txt}",
            f"  Εξίσωση ευθείας: {self.equation_str()}",
            f"  Μέσο: {self.midpoint('M')}",
        ]
        return "\n".join(lines)

    def plot(self, ax=None, color="tab:blue", show_points=True, show_label=True, arrow=False, **kwargs):
        """Σχεδιάζει το τμήμα. Με arrow=True δείχνει και τη φορά του."""
        if ax is None:
            _, ax = plt.subplots(figsize=(5, 5))
        if arrow:
            ax.annotate("", xy=self.end.to_tuple(), xytext=self.start.to_tuple(),
                        arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
        else:
            ax.plot([self.start.x, self.end.x], [self.start.y, self.end.y], color=color, **kwargs)
        if show_points:
            self.start.plot(ax, color=color, show_label=show_label)
            self.end.plot(ax, color=color, show_label=show_label)
        if show_label and self.name:
            m = self.midpoint()
            ax.annotate(self.name, m.to_tuple(), textcoords="offset points",
                        xytext=(-10, -12), fontsize=9, color=color)
        ax.set_aspect("equal")
        ax.grid(True, linestyle=":")
        ax.set_xlabel("X (Ανατολή)")
        ax.set_ylabel("Y (Βορράς)")
        return ax

### Παραδείγματα με `Line2D`

In [ ]:
AB = Line2D(A, B)          # το όνομα "AB" προκύπτει αυτόματα από τα σημεία
print(AB)
print()
print(AB.describe())

In [ ]:
C = Point2D(100.0, 260.0, "C")
D = Point2D(160.0, 190.0, "D")
CD = Line2D(C, D)

P = AB.intersection(CD, name="T")
print(f"Τομή AB και CD          : {P}")
print(f"Το T ανήκει στο AB;      {AB.contains_point(P)}")
print(f"Απόσταση C από το AB     : {AB.distance_to_point(C):.3f}")
print(f"AB παράλληλο στο CD;     {AB.is_parallel(CD)}")
print(f"Σημείο στο 25% του AB    : {AB.point_at(0.25)}")
print(f"Αντίστροφο τμήμα         : {AB.reversed()}  (αζιμούθιο {AB.reversed().azimuth():.4f} grad)")

ax = AB.plot(arrow=True)
CD.plot(ax, color="tab:orange", arrow=True)
P.plot(ax, color="black")
ax.set_title("Τομή δύο ευθύγραμμων τμημάτων")
plt.show()

## 3. Η κλάση `Polygon`

Ένα πολύγωνο ορίζεται από μια **λίστα κορυφών** (`Point2D`) με τη σειρά που τις διατρέχουμε. Το πολύγωνο κλείνει αυτόματα: η τελευταία κορυφή ενώνεται με την πρώτη. Αν η πρώτη κορυφή επαναληφθεί στο τέλος, η επανάληψη αφαιρείται.

Βασικές μέθοδοι:

| Μέθοδος | Τι κάνει |
|---|---|
| `edges()` | Λίστα με τις πλευρές ως `Line2D` |
| `perimeter()` | Περίμετρος |
| `signed_area()`, `area()` | Προσημασμένο και απόλυτο εμβαδόν (τύπος του Gauss) |
| `orientation()` | Φορά διαγραφής των κορυφών (δεξιόστροφη ή αριστερόστροφη) |
| `centroid()` | Κέντρο βάρους |
| `bounding_box()` | Ελάχιστο ορθογώνιο που το περικλείει |
| `interior_angles(unit)` | Εσωτερικές γωνίες σε κάθε κορυφή |
| `is_convex()` | Αν είναι κυρτό |
| `is_simple()` | Αν είναι απλό (οι πλευρές του δεν αυτοτέμνονται) |
| `contains_point(p)` | Αν ένα σημείο είναι μέσα στο πολύγωνο |
| `describe()`, `plot(ax)` | Περιγραφή και σχεδίαση |

**Εμβαδόν με τον τύπο του Gauss (shoelace):**

$$
E = \frac{1}{2} \sum_{i=1}^{n} \left( X_i\,Y_{i+1} - X_{i+1}\,Y_i \right)
$$

Αν $E > 0$, οι κορυφές διατρέχονται **αριστερόστροφα**. Αν $E < 0$, **δεξιόστροφα**. Το εμβαδόν του πολυγώνου είναι $|E|$.

**Κέντρο βάρους:**

$$
X_G = \frac{1}{6E} \sum_{i=1}^{n} (X_i + X_{i+1})(X_i Y_{i+1} - X_{i+1} Y_i), \qquad
Y_G = \frac{1}{6E} \sum_{i=1}^{n} (Y_i + Y_{i+1})(X_i Y_{i+1} - X_{i+1} Y_i)
$$

**Σημείο μέσα σε πολύγωνο (ray casting):** από το σημείο «ρίχνουμε» μια οριζόντια ημιευθεία και μετράμε πόσες πλευρές τέμνει. Αν τις τέμνει περιττό αριθμό φορών, το σημείο είναι μέσα.

In [ ]:
POLYGON_NAMES = {3: "Τρίγωνο", 4: "Τετράπλευρο", 5: "Πεντάγωνο", 6: "Εξάγωνο",
                 7: "Επτάγωνο", 8: "Οκτάγωνο", 9: "Εννεάγωνο", 10: "Δεκάγωνο"}


class Polygon:
    """Πολύγωνο που ορίζεται από μια λίστα κορυφών Point2D (κλείνει αυτόματα)."""

    def __init__(self, vertices, name=None):
        vertices = list(vertices)
        if not all(isinstance(v, Point2D) for v in vertices):
            raise TypeError("Όλες οι κορυφές πρέπει να είναι αντικείμενα Point2D.")
        # Αν ο χρήστης έκλεισε το πολύγωνο επαναλαμβάνοντας την πρώτη κορυφή, την αφαιρούμε
        if len(vertices) > 1 and vertices[0] == vertices[-1]:
            vertices = vertices[:-1]
        if len(vertices) < 3:
            raise ValueError("Ένα πολύγωνο χρειάζεται τουλάχιστον 3 διαφορετικές κορυφές.")
        self.vertices = vertices
        self.name = name
        if math.isclose(self.signed_area(), 0, abs_tol=TOL):
            raise ValueError("Το εμβαδόν είναι μηδέν: όλες οι κορυφές είναι συνευθειακές.")

    # ------------------------------------------------ Ειδικές μέθοδοι
    def __repr__(self):
        return f"Polygon(vertices={self.vertices!r}, name={self.name!r})"

    def __str__(self):
        label = self.name if self.name else "Πολύγωνο"
        return f"{label}: " + " → ".join(str(v) for v in self.vertices)

    def __len__(self):
        return len(self.vertices)

    def __iter__(self):
        return iter(self.vertices)

    def __getitem__(self, i):
        return self.vertices[i]

    # ------------------------------------------------ Βοηθητικά
    def _pairs(self):
        # Ζεύγη διαδοχικών κορυφών (Pi, Pi+1), μαζί με το (Pn, P1) που κλείνει το πολύγωνο
        n = len(self.vertices)
        return [(self.vertices[i], self.vertices[(i + 1) % n]) for i in range(n)]

    def type_name(self):
        n = len(self)
        return POLYGON_NAMES.get(n, f"{n}-γωνο")

    # ------------------------------------------------ Πλευρές και μήκη
    def edges(self):
        """Λίστα με τις πλευρές του πολυγώνου ως Line2D."""
        return [Line2D(p, q) for p, q in self._pairs()]

    def perimeter(self):
        return sum(e.length for e in self.edges())

    # ------------------------------------------------ Εμβαδόν και φορά
    def signed_area(self):
        """Τύπος του Gauss: θετικό για αριστερόστροφη και αρνητικό για δεξιόστροφη φορά."""
        return 0.5 * sum(p.x * q.y - q.x * p.y for p, q in self._pairs())

    def area(self):
        return abs(self.signed_area())

    def orientation(self):
        return "αριστερόστροφη (CCW)" if self.signed_area() > 0 else "δεξιόστροφη (CW)"

    def is_ccw(self):
        return self.signed_area() > 0

    # ------------------------------------------------ Κέντρο βάρους και όρια
    def centroid(self, name="G"):
        E = self.signed_area()
        cx = sum((p.x + q.x) * (p.x * q.y - q.x * p.y) for p, q in self._pairs()) / (6 * E)
        cy = sum((p.y + q.y) * (p.x * q.y - q.x * p.y) for p, q in self._pairs()) / (6 * E)
        return Point2D(cx, cy, name)

    def bounding_box(self):
        """Επιστρέφει (Xmin, Ymin, Xmax, Ymax)."""
        xs = [v.x for v in self.vertices]
        ys = [v.y for v in self.vertices]
        return min(xs), min(ys), max(xs), max(ys)

    # ------------------------------------------------ Γωνίες και μορφή
    def interior_angles(self, unit="grad"):
        """Εσωτερική γωνία σε κάθε κορυφή (ισχύει και για μη κυρτά πολύγωνα)."""
        n = len(self)
        angles = []
        for i in range(n):
            prev_v = self.vertices[i - 1]
            cur = self.vertices[i]
            next_v = self.vertices[(i + 1) % n]
            a_prev = math.atan2(prev_v.y - cur.y, prev_v.x - cur.x)
            a_next = math.atan2(next_v.y - cur.y, next_v.x - cur.x)
            diff = (a_prev - a_next) if self.is_ccw() else (a_next - a_prev)
            angles.append(convert_angle(diff % (2 * math.pi), unit))
        return angles

    def is_convex(self):
        """Κυρτό: όλα τα διαδοχικά εξωτερικά γινόμενα έχουν το ίδιο πρόσημο."""
        n = len(self)
        signs = set()
        for i in range(n):
            p0, p1, p2 = self.vertices[i], self.vertices[(i + 1) % n], self.vertices[(i + 2) % n]
            c = cross(p1.x - p0.x, p1.y - p0.y, p2.x - p1.x, p2.y - p1.y)
            if not math.isclose(c, 0, abs_tol=TOL):
                signs.add(c > 0)
        return len(signs) == 1

    def is_simple(self):
        """Απλό πολύγωνο: δεν τέμνονται πλευρές που δεν είναι διαδοχικές."""
        edges = self.edges()
        n = len(edges)
        for i in range(n):
            for j in range(i + 1, n):
                # Οι διαδοχικές πλευρές μοιράζονται κορυφή, οπότε τις παραλείπουμε
                if j == i + 1 or (i == 0 and j == n - 1):
                    continue
                if edges[i].intersection(edges[j]) is not None:
                    return False
        return True

    # ------------------------------------------------ Σημείο μέσα στο πολύγωνο
    def contains_point(self, p, include_boundary=True):
        """Ray casting. Τα σημεία πάνω στις πλευρές θεωρούνται μέσα αν include_boundary=True."""
        if any(e.contains_point(p) for e in self.edges()):
            return include_boundary
        inside = False
        for a, b in self._pairs():
            if (a.y > p.y) != (b.y > p.y):  # η πλευρά «διασχίζει» την οριζόντια ευθεία του p
                x_cross = a.x + (p.y - a.y) * (b.x - a.x) / (b.y - a.y)
                if x_cross > p.x:
                    inside = not inside
        return inside

    # ------------------------------------------------ Περιγραφή και σχεδίαση
    def describe(self, unit="grad"):
        """Επιστρέφει πλήρη περιγραφή του πολυγώνου σε κείμενο."""
        sym = "grad" if unit == "grad" else ("°" if unit == "deg" else "rad")
        xmin, ymin, xmax, ymax = self.bounding_box()
        angles = self.interior_angles(unit)
        full_turn = convert_angle(math.pi, unit)  # 200 grad ή 180°
        expected = (len(self) - 2) * full_turn
        lines = [
            f"Πολύγωνο: {self.name if self.name else '(χωρίς όνομα)'}",
            f"  Τύπος: {self.type_name()} ({len(self)} κορυφές)",
            f"  Φορά κορυφών: {self.orientation()}",
            f"  Κυρτό: {'ναι' if self.is_convex() else 'όχι'}",
            f"  Απλό (χωρίς αυτοτομές): {'ναι' if self.is_simple() else 'ΟΧΙ: οι πλευρές τέμνονται!'}",
            f"  Περίμετρος: {self.perimeter():.3f}",
            f"  Εμβαδόν: {self.area():.3f}",
            f"  Κέντρο βάρους: {self.centroid()}",
            f"  Περιβάλλον ορθογώνιο: X ∈ [{xmin:.3f}, {xmax:.3f}], Y ∈ [{ymin:.3f}, {ymax:.3f}]",
            "  Κορυφές:",
        ]
        for v, ang in zip(self.vertices, angles):
            lines.append(f"    {str(v):<28} εσωτ. γωνία = {ang:9.4f} {sym}")
        lines.append("  Πλευρές:")
        for e in self.edges():
            label = e.name if e.name else "-"
            lines.append(f"    {label:<6} μήκος = {e.length:10.3f}   αζιμούθιο = {e.azimuth():9.4f} grad")
        lines.append(f"  Άθροισμα εσωτ. γωνιών: {sum(angles):.4f} {sym}"
                     f" (θεωρητικά (n−2)·{full_turn:.0f} = {expected:.0f} {sym})")
        if not self.is_simple():
            lines.append("  ΠΡΟΣΟΧΗ: το πολύγωνο αυτοτέμνεται, οπότε το εμβαδόν, το κέντρο βάρους"
                         " και οι γωνίες δεν έχουν τη συνηθισμένη γεωμετρική σημασία.")
        return "\n".join(lines)

    def plot(self, ax=None, color="tab:green", alpha=0.3, show_labels=True, show_centroid=True):
        if ax is None:
            _, ax = plt.subplots(figsize=(6, 6))
        xs = [v.x for v in self.vertices] + [self.vertices[0].x]
        ys = [v.y for v in self.vertices] + [self.vertices[0].y]
        ax.fill(xs, ys, color=color, alpha=alpha)
        ax.plot(xs, ys, color=color, lw=1.5)
        for v in self.vertices:
            v.plot(ax, color=color, show_label=show_labels)
        if show_centroid:
            self.centroid().plot(ax, color="black", marker="x", show_label=show_labels)
        if self.name:
            ax.set_title(self.name)
        ax.set_aspect("equal")
        ax.grid(True, linestyle=":")
        ax.set_xlabel("X (Ανατολή)")
        ax.set_ylabel("Y (Βορράς)")
        return ax

### Παραδείγματα με `Polygon`

Πρώτα ένα απλό τετράπλευρο, για το οποίο υπολογίζουμε εύκολα το εμβαδόν και με το χέρι.

In [ ]:
square = Polygon([Point2D(0, 0, "1"), Point2D(10, 0, "2"),
                  Point2D(10, 10, "3"), Point2D(0, 10, "4")], name="Τετράγωνο")
print(square.describe())

Στη συνέχεια ένα **μη κυρτό** πολύγωνο (σχήμα «Γ»), δοσμένο με δεξιόστροφη φορά, και έλεγχος για το αν κάποια σημεία βρίσκονται μέσα σε αυτό.

In [ ]:
parcel = Polygon([
    Point2D(200, 500, "K1"),
    Point2D(260, 500, "K2"),
    Point2D(260, 440, "K3"),
    Point2D(320, 440, "K4"),
    Point2D(320, 400, "K5"),
    Point2D(200, 400, "K6"),
], name="Οικόπεδο σε σχήμα Γ")

print(parcel.describe(unit="deg"))

In [ ]:
test_points = [Point2D(230, 470, "Σ1"), Point2D(290, 470, "Σ2"),
               Point2D(300, 420, "Σ3"), Point2D(260, 460, "Σ4")]

for p in test_points:
    print(f"{p}: {'μέσα' if parcel.contains_point(p) else 'έξω'}")

ax = parcel.plot()
for p in test_points:
    p.plot(ax, color="tab:green" if parcel.contains_point(p) else "tab:red")
plt.show()

Τέλος, ένα πολύγωνο του οποίου οι πλευρές **αυτοτέμνονται** («παπιγιόν»). Η `describe()` το επισημαίνει, επειδή για τέτοια πολύγωνα το εμβαδόν του Gauss δεν έχει τη συνηθισμένη γεωμετρική σημασία.

In [ ]:
bowtie = Polygon([Point2D(0, 0, "A"), Point2D(6, 4, "B"),
                  Point2D(6, 0, "C"), Point2D(0, 6, "D")], name="Παπιγιόν")
print(bowtie.describe())
bowtie.plot(color="tab:red")
plt.show()

## 4. Έλεγχος λανθασμένων δεδομένων

Οι κλάσεις ελέγχουν τα δεδομένα που τους δίνουμε και εμφανίζουν κατανοητά μηνύματα σφάλματος, αντί να επιστρέφουν λάθος αποτελέσματα χωρίς προειδοποίηση.

In [ ]:
bad_inputs = [
    ("Τμήμα με ίδια άκρα",        lambda: Line2D(Point2D(1, 1), Point2D(1, 1))),
    ("Τμήμα με άκρα που δεν είναι σημεία", lambda: Line2D((0, 0), (1, 1))),
    ("Πολύγωνο με 2 κορυφές",     lambda: Polygon([Point2D(0, 0), Point2D(1, 1)])),
    ("Πολύγωνο με συνευθειακές κορυφές", lambda: Polygon([Point2D(0, 0), Point2D(1, 1), Point2D(2, 2)])),
    ("Αζιμούθιο προς το ίδιο σημείο", lambda: A.azimuth_to(A)),
]

for title, func in bad_inputs:
    try:
        func()
        print(f"{title:<36} → κανένα σφάλμα")
    except (ValueError, TypeError) as err:
        print(f"{title:<36} → {type(err).__name__}: {err}")

## 5. Όλα μαζί σε ένα σχήμα

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_parcel = Polygon([Point2D(100, 180, "Π1"), Point2D(170, 190, "Π2"),
                       Point2D(180, 250, "Π3"), Point2D(120, 270, "Π4")], name="Πολύγωνο Π")
plot_parcel.plot(ax)
AB.plot(ax, color="tab:blue", arrow=True)
CD.plot(ax, color="tab:orange", arrow=True)

# Τομές των τμημάτων AB και CD με τις πλευρές του πολυγώνου
for line in (AB, CD):
    for edge in plot_parcel.edges():
        t = line.intersection(edge)
        if t is not None:
            t.plot(ax, color="black", show_label=False, marker="s")

ax.set_title("Σημεία, τμήματα και πολύγωνο")
plt.show()